### Building a RAG System with LangChain and FAISS 
Introduction to RAG (Retrieval-Augmented Generation)
RAG combines the power of retrieval systems with generative AI models. Instead of relying solely on the model's training data, RAG:

1. Retrieves relevant documents from a knowledge base
2. Uses these documents as context for the LLM
3. Generates responses based on both the retrieved context and the model's knowledge

### FAISS 
https://github.com/facebookresearch/faiss

FAISS is a library for efficient similarity search and clustering of dense vectors.

Key advantages:
1. Extremely fast similarity search
2. Memory efficient
3. Supports GPU acceleration
4. Can handle millions of vectors

How it works:
- Indexes vectors for fast nearest neighbor search
- Returns most similar vectors based on distance metrics


In [2]:
## load libraries
import os
from dotenv import load_dotenv
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# LangChain core imports
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import (
    RunnablePassthrough, 
 
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# LangChain specific imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Load environment variables
load_dotenv()

True

### Data Ingestion And Processing


In [3]:
sample_documents = [
    Document(
        page_content="""
        Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
        """,
        metadata={"source": "AI Introduction", "page": 1, "topic": "AI"}
    ),
    Document(
        page_content="""
        Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.
        """,
        metadata={"source": "ML Basics", "page": 1, "topic": "ML"}
    ),
    Document(
        page_content="""
        Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition.
        """,
        metadata={"source": "Deep Learning", "page": 1, "topic": "DL"}
    ),
    Document(
        page_content="""
        Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
        Applications include chatbots, translation, sentiment analysis, and text summarization.
        """,
        metadata={"source": "NLP Overview", "page": 1, "topic": "NLP"}
    )
]

print(sample_documents)

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='\n        Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.\n        '), Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='\n        Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.\n        '), Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='\n        Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolu

In [6]:
## Text splitting
text_splitter= RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=[" "]
)

## split the documents into chunks
chunks = text_splitter.split_documents(sample_documents)
print(chunks[0])
print(chunks[1])

page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.' metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}
page_content='Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.' metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}


In [7]:
print(f"Created {len(chunks)} chunks from {len(sample_documents)} documents")
print("\nExample chunk:")
print(f"Content: {chunks[0].page_content}")
print(f"Metadata: {chunks[0].metadata}")

Created 4 chunks from 4 documents

Example chunk:
Content: Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
Metadata: {'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}


In [10]:
## Load the embeddding models
import os


In [13]:
# Initialize embeddings with the latest model

from langchain_huggingface import HuggingFaceEmbeddings


embeddings= HuggingFaceEmbeddings(
    model_name ="sentence-transformers/all-MiniLM-L6-v2"
)
embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4651.81it/s]


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [14]:
## Example: Create a embedding for a single text
sample_text = "What is machine learning"
sample_embedding= embeddings.embed_query(sample_text)
sample_embedding

[-0.029035668820142746,
 0.007559695281088352,
 0.040764763951301575,
 0.030535506084561348,
 0.05175705999135971,
 -0.01734757423400879,
 -0.030941858887672424,
 -0.06556294113397598,
 -0.03306078165769577,
 -0.009561226703226566,
 -0.09131385385990143,
 0.0473836287856102,
 0.02773956209421158,
 -0.06366026401519775,
 -0.0650474950671196,
 0.04200262576341629,
 -0.0402190163731575,
 0.028005139902234077,
 -0.028049848973751068,
 -0.053762827068567276,
 -0.0054283132776618,
 0.006084789987653494,
 -0.07086586207151413,
 0.025930071249604225,
 0.01007944718003273,
 0.026293473318219185,
 0.03857467696070671,
 0.02275247685611248,
 -0.016700468957424164,
 0.009418035857379436,
 0.01637609675526619,
 -0.05924321711063385,
 -0.01635242998600006,
 0.04579901695251465,
 -0.06120172142982483,
 0.06117105856537819,
 -0.013366833329200745,
 -0.00031105318339541554,
 0.03935341536998749,
 -0.04183914512395859,
 -0.03927674889564514,
 -0.1010734960436821,
 -0.006085134577006102,
 -0.011100508272

In [15]:
texts = ["AI", "Machine Learning", "Deep Learning", "Neural Network"]
batch_embeddings = embeddings.embed_documents(texts)
print(batch_embeddings[0])

[-0.03653920069336891, -0.0151643892750144, 0.016432467848062515, 0.010568763129413128, 0.006010628305375576, -0.018473191186785698, 0.085465207695961, 0.02096848376095295, 0.027815336361527443, 0.012431707233190536, -0.029376493766903877, -0.03113521821796894, 0.034912582486867905, -0.018150869756937027, -0.06498478353023529, 0.05168255791068077, -0.019606169313192368, -0.015734177082777023, -0.13371670246124268, -0.09645996242761612, -0.025471772998571396, -0.0014895793283358216, -0.006349316798150539, -0.02582070417702198, -0.027371713891625404, 0.12268999218940735, -0.007792447227984667, -0.03852275013923645, 0.014383506029844284, -0.09218421578407288, 0.008695731870830059, 0.0026133290957659483, 0.09103468060493469, -0.030313584953546524, -0.09604635834693909, 0.022289108484983444, -0.09024312347173691, -0.032947391271591187, 0.07158336788415909, -0.00889311358332634, -0.025708945468068123, -0.07913963496685028, 0.014530414715409279, -0.07420432567596436, 0.08045010268688202, 0.07

In [16]:
print(batch_embeddings[1])

[-0.02439182810485363, 0.0032444624230265617, 0.054267626255750656, -0.006672567687928677, 0.003935707733035088, -0.007957384921610355, 0.025025270879268646, -0.03203270584344864, -0.054510697722435, -0.044702015817165375, -0.013759400695562363, 0.016061272472143173, 0.04036473482847214, -0.020260926336050034, -0.06097456440329552, 0.020655646920204163, 0.010556362569332123, -0.016264762729406357, -0.1049070954322815, -0.1106831282377243, -0.021544644609093666, -0.013036168180406094, -0.08688357472419739, 0.027151908725500107, 0.0261439960449934, 0.03964658081531525, 0.06494349986314774, 0.0654725730419159, 0.017963245511054993, -0.10655660927295685, 0.009878205135464668, -0.034962043166160583, 0.030403533950448036, 0.014532789587974548, -0.11560273170471191, 0.012346271425485611, -0.0643094852566719, 0.043945975601673126, 0.019033070653676987, 0.03098483197391033, -0.015413939021527767, -0.08163458108901978, 0.0124144796282053, 0.012423684820532799, 0.06950362026691437, 0.077825509011

In [17]:
## Compare Embedding using cosine similarity

def compare_embeddings(text1:str, text2:str):
    """Compare semantic similarity of 2 texts using embeddings"""

    emb1 = np.array(embeddings.embed_query(text1))
    emb2 = np.array(embeddings.embed_query(text2))

    ## Calculate the similarity score

    similarity = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
    return similarity

In [18]:
## Test semantic similarity
print("\nSemantic Similarity Example: ")
print(f"'AI' vs 'Artificial Intelligence: {compare_embeddings('AI', 'Artificial Intelligence'): .3f}")


Semantic Similarity Example: 
'AI' vs 'Artificial Intelligence:  0.791


In [19]:
print(f"'AI' vs 'Pizza': {compare_embeddings('AI', 'Pizza'):.3f}")

'AI' vs 'Pizza': 0.257


In [20]:
print(f"'Machine Learning' vs 'ML': {compare_embeddings('Machine Learning', 'ML'):.3f}")

'Machine Learning' vs 'ML': 0.373


### Create FAISS Vector Store

In [22]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)
print(f"Vector store created with {vectorstore.index.ntotal} vectors")

Vector store created with 4 vectors


In [23]:
vectorstore

In [24]:
## Save vector store for later use
vectorstore.save_local("faiss_index")
print("Vector store saved to 'faiss_index' successfully")

Vector store saved to 'faiss_index' successfully


In [25]:
## Load vector store
loaded_vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

print(f"Loaded vector store contains {loaded_vectorstore.index.ntotal} vectors")

Loaded vector store contains 4 vectors


In [26]:
## Similarity search
query = "What is deep learning"

results = vectorstore.similarity_search(query,k=3)
print(results)


[Document(id='b28e2157-9654-4ed4-8621-caa23d33e6b6', metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolutionized computer vision, NLP, and speech recognition.'), Document(id='6293b937-1b43-4aeb-9fd9-0e929d2e3959', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'), Document(id='426ce315-abe0-4a46-9ae7-76070e2fa240', metadata={'source': 'NLP Overview', 'page': 1, 'topic': 'NLP'}, page_content='Natural Language Processing (NLP) is a branch of AI that helps computers understand human lang

In [27]:
print(f"Query: {query}\n")
print("Top 3 similar chunks:")
for i, doc in enumerate(results):
    print(f"\n{i+1}. Source: {doc.metadata['source']}")
    print(f" Content: {doc.page_content[:200]}...")

Query: What is deep learning

Top 3 similar chunks:

1. Source: Deep Learning
 Content: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning ...

2. Source: ML Basics
 Content: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised...

3. Source: NLP Overview
 Content: Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
      ...


In [28]:
## Similarity Search with score
result_with_score = vectorstore.similarity_search_with_score(query,k=3)

print("\n\nSimilarity search with scores.")
for doc, score in result_with_score:
    print(f"\nScore: {score:.3f}")
    print(f"Source: {doc.metadata['source']}")
    print(f"Content preview: {doc.page_content[:100]}")



Similarity search with scores.

Score: 0.343
Source: Deep Learning
Content preview: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses m

Score: 1.090
Source: ML Basics
Content preview: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being

Score: 1.154
Source: NLP Overview
Content preview: Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.



In [29]:
chunks

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.'),
 Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'),
 Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolutionized computer vision, NLP, and speech recogn

In [30]:
## Search with metadata filtering
filter_dict = {"topic":"ML"}
filtered_results = vectorstore.similarity_search(
    query,
    k=3,
    filter= filter_dict
)
print(filtered_results)

[Document(id='6293b937-1b43-4aeb-9fd9-0e929d2e3959', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.')]


In [31]:
len(filtered_results)

1

### Build RAG Chain With LCEL 

In [32]:
## LLM GQOK
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

llm= ChatGroq(
    model_name = "openai/gpt-oss-120b"
)

In [33]:
llm.invoke("hi")

AIMessage(content='Hey! What’s up? 😊', additional_kwargs={'reasoning_content': 'The user says "hi". The system instructions: we must follow the system prompt: "You are ChatGPT, ... You are ChatGPT, a large language model trained by OpenAI. ... You must respond in the style of a 15-year-old high school student from the US." Also must obey developer message: "You are ChatGPT..." The user just says hi. We respond in a teenage style. Probably "Hey! What\'s up?" etc. Should be short.'}, response_metadata={'token_usage': {'completion_tokens': 113, 'prompt_tokens': 72, 'total_tokens': 185, 'completion_time': 0.236206882, 'completion_tokens_details': {'reasoning_tokens': 97}, 'prompt_time': 0.003344191, 'prompt_tokens_details': None, 'queue_time': 0.212121488, 'total_time': 0.239551073}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4a7bb72c24', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a11fc5-d67c-79d3-813

In [34]:
# 1. Simple RAG Chain with LCEL
simple_prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context:
Context: {context}

Question: {question}

Answer:""")

In [35]:
## Basic retriever
retriever= vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs = {"k":3}
)

In [36]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001716F993CB0>, search_kwargs={'k': 3})

In [37]:
from typing import List

# Format documents for the prompt
def format_docs(docs: List[Document]) -> str:
    """Format documents for insertion into prompt"""
    formatted = []
    for i,doc in enumerate(docs):
        source = doc.metadata.get('source','Unknown')
        formatted.append(f"Document {i+1} (Source: {source}): \n{doc.page_content}")
    return "\n\n".join(formatted)


In [39]:
simple_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough() }
    | simple_prompt
    | llm
    |StrOutputParser()
)

In [40]:
simple_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\nContext: {context}\n\nQuestion: {question}\n\nAnswer:'), additional_kwargs={})])

In [ ]:
### Conversational RAG Chain

conversational_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Use the provided context to answer questions."),
    ("placeholder", "{chat_history}"),
    ("human", "Context: {context}\n\nQuestion: {input}"),
])

In [42]:
def create_conversational_rag():
    """Create a conversational RAG chain with memory"""
    return (
        RunnablePassthrough.assign(
            context= lambda x: format_docs(retriever.invoke(x["input"]))
        )
        | conversational_prompt
        | llm
        | StrOutputParser()
    )

conversational_rag = create_conversational_rag()

In [43]:
conversational_rag

RunnableAssign(mapper={
  context: RunnableLambda(lambda x: format_docs(retriever.invoke(x['input'])))
})
| ChatPromptTemplate(input_variables=['context', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag=

In [44]:
## Streaming RAG Chain
streaming_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | simple_prompt
    | llm
)

print("Modern RAG chain created successfully!")
print("Available chains: ")
print("- simple_rag_chain: Basic Q&A")
print("- conversational_rag: Maintains conversation history")
print("- streaming_rag_chain: Supports token streaming")

Modern RAG chain created successfully!
Available chains: 
- simple_rag_chain: Basic Q&A
- conversational_rag: Maintains conversation history
- streaming_rag_chain: Supports token streaming


In [45]:
## Test function for different chain types
def test_rag_chains(question: str):
    """Test all RAG chain variants"""
    print(f"Question: {question}")
    print("=" * 80)

    # 1. Simple RAG
    print("\n1. Simple RAG Chain: ")
    answer = simple_rag_chain.invoke(question)
    print(f"Answer: {answer}")

    print("\n2. Streaming RAG: ")
    print("Answer: ", end="", flush=True)
    for chunk in streaming_rag_chain.stream(question):
        print(chunk.content, end="", flush=True)
    print()

In [46]:
test_rag_chains("What is the difference between AI and machine Learning")

Question: What is the difference between AI and machine Learning

1. Simple RAG Chain: 
Answer: **Difference between AI and Machine Learning (based on the provided context)**  

- **Artificial Intelligence (AI)** is the broader field that aims to simulate human intelligence in machines. It encompasses any system designed to think like humans and mimic human actions, and it can be divided into **narrow AI** (focused on specific tasks) and **general AI** (capable of broader, human‑like intelligence).  

- **Machine Learning (ML)** is a **subset of AI**. It specifically enables systems to **learn from data** rather than being explicitly programmed. ML algorithms discover patterns in data and include approaches such as supervised, unsupervised, and reinforcement learning.  

**In short:** AI is the overall discipline of creating intelligent behavior, while Machine Learning is one particular way to achieve that intelligence—by training models on data to automatically find patterns and make 

In [47]:
## Test with multiple questions
test_questions = [
    "What is the difference between AI and machine learning?",
    "Explain deep learning in simple terms",
    "How does NLP work?"
]

for question in test_questions:
    print("\n" + "=" * 80 + "\n")
    test_rag_chains(question)



Question: What is the difference between AI and machine learning?

1. Simple RAG Chain: 
Answer: AI (Artificial Intelligence) is the broad field that aims to simulate human intelligence in machines, enabling them to think and act like humans. Machine learning (ML) is a specific subset of AI; it focuses on giving systems the ability to learn from data rather than being explicitly programmed, allowing them to discover patterns and improve performance on tasks. In short, AI encompasses the overall goal of intelligent behavior, while machine learning is one way to achieve that goal by using data‑driven learning algorithms.

2. Streaming RAG: 
Answer: Artificial Intelligence (AI) is the broad field that aims to simulate human intelligence in machines, encompassing any technique that enables computers to think or act like humans.  

Machine Learning (ML) is a **subset of AI**. It specifically refers to methods that allow systems to learn from data—discovering patterns and making decisions—

In [50]:
## Conversational example
print("\n3. Conversational RAG example:")
chat_history = []

# First question
q1 = "What is machine learning?"
a1 = conversational_rag.invoke({
    "input": q1,
    "chat_history": chat_history
})

print(f"Q1: {q1}")
print(f"Q2: {a1}")


3. Conversational RAG example:
Q1: What is machine learning?
Q2: Machine learning (ML) is a branch of artificial intelligence that enables systems to learn directly from data rather than being explicitly programmed. By analyzing data, ML algorithms discover patterns and build models that can make predictions or decisions on new, unseen inputs.


In [51]:
# Update history
chat_history.extend([
    HumanMessage(content=q1),
    AIMessage(content=a1)
])

In [52]:
## Follow-up question
q2 = "How is it different from traditional programming?"
a2 = conversational_rag.invoke({
    "input": q2,
    "chat_history": chat_history
})
print(f"\nQ2: {q2}")
print(f"A2: {a2}")


Q2: How is it different from traditional programming?
A2: **Machine learning vs. traditional (rule‑based) programming**

| Aspect | Traditional Programming | Machine‑Learning‑Based Systems |
|--------|--------------------------|--------------------------------|
| **How behavior is created** | The programmer writes explicit **if‑then rules** or algorithms that dictate exactly what the program should do for every possible input. | The programmer writes a **learning algorithm** and supplies data; the system **infers** the rules/patterns from that data. |
| **Source of knowledge** | Knowledge is encoded manually by developers (e.g., “if temperature > 30 °C then turn on fan”). | Knowledge is extracted automatically from examples (e.g., a model learns that “high temperature + high humidity” often means “turn on fan”). |
| **Adaptability** | Changing behavior requires **rewriting or adding code**; the program does not adapt on its own. | The model can be **re‑trained** with new data and will